# Phase-3 Step 3: FastAPI Deployment

**Objective:** Deploy the SOH prediction model as a REST API.

**Tasks:**
1. Save production model
2. Test API locally
3. Docker containerization
4. Example client code

In [ ]:
# --- Portable setup (Windows / macOS / Linux) — do not hardcode paths ---
import importlib.util
from pathlib import Path

def _load_notebook_setup():
    cwd = Path.cwd().resolve()
    candidates = [
        cwd / "notebooks" / "notebook_setup.py",
        cwd / "notebook_setup.py",
    ]
    if cwd.name == "notebooks":
        candidates.insert(0, cwd / "notebook_setup.py")
    for fp in candidates:
        if fp.is_file():
            spec = importlib.util.spec_from_file_location("_notebook_setup", fp)
            mod = importlib.util.module_from_spec(spec)
            assert spec.loader is not None
            spec.loader.exec_module(mod)
            return mod
    raise FileNotFoundError(
        "Missing notebooks/notebook_setup.py. Unzip the full project, open a terminal, "
        "cd into the repo root, then run: jupyter notebook"
    )

_ns = _load_notebook_setup()
PROJECT_ROOT = _ns.setup(with_api=True)
print(f"PROJECT_ROOT = {PROJECT_ROOT}")


import sys
import os
import warnings
warnings.filterwarnings('ignore')



import numpy as np
import pandas as pd
import pickle

print("✅ Setup complete")

## 1. Train and Save Production Model

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from data_splits import create_within_battery_split, normalize_features

# Load data
all_features = pd.read_csv('data/phase2/harmonized/all_features.csv')
print(f"Loaded {len(all_features)} samples")

# Create splits
splits = create_within_battery_split(all_features, train_ratio=0.7, val_ratio=0.15)

# Feature columns
exclude_cols = ['battery_id', 'cycle_number', 'original_index', 'soh', 'has_spikes']
feature_cols = [c for c in all_features.columns if c not in exclude_cols]

print(f"Features: {len(feature_cols)}")

In [ ]:
# Combine train + val for final model
train_df = pd.concat([splits['train'], splits['val']], ignore_index=True)
test_df = splits['test']

# Fit scaler on training data
scaler = StandardScaler()
X_train = train_df[feature_cols].values
y_train = train_df['soh'].values

X_train_scaled = scaler.fit_transform(X_train)

X_test = test_df[feature_cols].values
y_test = test_df['soh'].values
X_test_scaled = scaler.transform(X_test)

print(f"Training samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")

In [ ]:
# Train final model
model = Ridge(alpha=1.0)
model.fit(X_train_scaled, y_train)

# Evaluate
y_pred = model.predict(X_test_scaled)

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"\n📊 Production Model Metrics:")
print(f"   MAE:  {mae:.4f}")
print(f"   RMSE: {rmse:.4f}")
print(f"   R²:   {r2:.4f}")

In [ ]:
# Save model and scaler
os.makedirs('models', exist_ok=True)

with open('models/best_model.pkl', 'wb') as f:
    pickle.dump(model, f)

with open('models/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

# Save feature columns
with open('models/feature_cols.pkl', 'wb') as f:
    pickle.dump(feature_cols, f)

print("✅ Saved:")
print("   models/best_model.pkl")
print("   models/scaler.pkl")
print("   models/feature_cols.pkl")

## 2. Test API Code

In [ ]:
# Test the API prediction function
# Simulate API prediction
def test_prediction(features_dict):
    """Test prediction with saved model."""
    # Load model
    with open('models/best_model.pkl', 'rb') as f:
        model = pickle.load(f)
    with open('models/scaler.pkl', 'rb') as f:
        scaler = pickle.load(f)
    with open('models/feature_cols.pkl', 'rb') as f:
        feature_cols = pickle.load(f)
    
    # Prepare features
    X = np.array([[features_dict.get(f, 0.0) for f in feature_cols]])
    X_scaled = scaler.transform(X)
    
    # Predict
    soh = model.predict(X_scaled)[0]
    return max(0, min(100, soh))

# Test with a sample
sample = test_df[feature_cols].iloc[0].to_dict()
actual = test_df['soh'].iloc[0]
predicted = test_prediction(sample)

print(f"\n🧪 Test Prediction:")
print(f"   Actual:    {actual:.2f}%")
print(f"   Predicted: {predicted:.2f}%")
print(f"   Error:     {abs(actual - predicted):.2f}%")

## 3. Install API Dependencies

In [ ]:
# Check if FastAPI is installed
try:
    import fastapi
    import uvicorn
    print(f"✅ FastAPI version: {fastapi.__version__}")
except ImportError:
    print("❌ FastAPI not installed")
    print("Run: pip install fastapi uvicorn[standard]")

## 4. Start API Server

Open a **terminal** and `cd` to the **project root** (folder containing `api/`, `src/`, `config.yaml`). The first code cell prints `PROJECT_ROOT`.

```bash
# Windows: cd C:\\path\\to\\repo
# macOS/Linux: cd /path/to/repo
source .venv/bin/activate
uvicorn api.main:app --reload --port 8000
```

On Windows use `.venv\\Scripts\\activate` instead of `source`.

Then visit: http://localhost:8000/docs

## 5. Test API Endpoints

In [ ]:
# Test API with requests (run after starting server)
import requests

API_URL = "http://localhost:8000"

# Health check
try:
    response = requests.get(f"{API_URL}/health", timeout=5)
    if response.ok:
        print("✅ API is running!")
        print(f"   Status: {response.json()}")
    else:
        print(f"❌ API returned: {response.status_code}")
except requests.exceptions.ConnectionError:
    print("❌ Cannot connect to API. Make sure the server is running.")
    print("   Run: uvicorn api.main:app --reload --port 8000")

In [ ]:
# Test prediction endpoint
try:
    # Prepare test data
    test_features = {
        "voltage_mean": 3.5,
        "voltage_std": 0.2,
        "voltage_min": 3.0,
        "voltage_max": 4.2,
        "voltage_range": 1.2,
        "current_mean": 1.5,
        "current_std": 0.1,
        "capacity": 1.8,
        "discharge_time": 3600
    }
    
    response = requests.post(
        f"{API_URL}/predict",
        json={"features": test_features, "include_explanation": True},
        timeout=10
    )
    
    if response.ok:
        result = response.json()
        print("\n✅ Prediction successful!")
        print(f"   SOH: {result['soh']:.2f}%")
        print(f"   Time: {result['processing_time_ms']:.2f} ms")
        if result.get('explanation'):
            print(f"   Explanation: {result['explanation']}")
    else:
        print(f"❌ Prediction failed: {response.status_code}")
        print(f"   {response.json()}")
except requests.exceptions.ConnectionError:
    print("❌ Cannot connect to API")

## 6. Docker Deployment

In [ ]:
# View Dockerfile
print("📦 Dockerfile:")
print("="*50)

with open('api/Dockerfile', 'r') as f:
    print(f.read())

In [ ]:
print("""
📦 Docker Commands:

# Build image
docker build -t battery-soh-api -f api/Dockerfile .

# Run container
docker run -d -p 8000:8000 --name soh-api battery-soh-api

# Check logs
docker logs soh-api

# Stop
docker stop soh-api && docker rm soh-api
""")

## 7. Example Client Code

In [ ]:
# Save example client code
client_code = '''
#!/usr/bin/env python3
"""
Example client for Battery SOH Prediction API
"""

import requests
import numpy as np

API_URL = "http://localhost:8000"

def predict_from_cycle(voltage, current, temperature=None):
    """
    Predict SOH from raw cycle data.
    
    Args:
        voltage: List of voltage measurements (V)
        current: List of current measurements (A)
        temperature: Optional list of temperature measurements (C)
    
    Returns:
        Predicted SOH (%)
    """
    payload = {
        "cycle_data": {
            "voltage": list(voltage),
            "current": list(current),
            "temperature": list(temperature) if temperature else None
        },
        "include_explanation": True
    }
    
    response = requests.post(f"{API_URL}/predict", json=payload)
    
    if response.ok:
        return response.json()
    else:
        raise Exception(f"API error: {response.status_code} - {response.text}")


def predict_from_features(features):
    """
    Predict SOH from pre-extracted features.
    
    Args:
        features: Dict with feature values
    
    Returns:
        Predicted SOH (%)
    """
    payload = {
        "features": features,
        "include_explanation": True
    }
    
    response = requests.post(f"{API_URL}/predict", json=payload)
    
    if response.ok:
        return response.json()
    else:
        raise Exception(f"API error: {response.status_code} - {response.text}")


if __name__ == "__main__":
    # Example 1: From features
    result = predict_from_features({
        "voltage_mean": 3.5,
        "voltage_std": 0.2,
        "capacity": 1.8,
        "discharge_time": 3600
    })
    
    print(f"Predicted SOH: {result[\'soh\']:.1f}%")
    print(f"Explanation: {result[\'explanation\']}")
    
    # Example 2: From raw cycle
    voltage = np.linspace(4.2, 3.0, 100).tolist()
    current = [-1.5] * 100
    
    result = predict_from_cycle(voltage, current)
    print(f"Predicted SOH: {result[\'soh\']:.1f}%")
'''

with open('api/client_example.py', 'w') as f:
    f.write(client_code)

print("✅ Saved: api/client_example.py")

In [ ]:
print("\n" + "="*70)
print("STEP 3 SUMMARY: DEPLOYMENT")
print("="*70)

print("""
✅ Completed:
   1. Trained and saved production model
   2. Created FastAPI server (api/main.py)
   3. Created Dockerfile for containerization
   4. Created example client code

📋 Files created:
   - models/best_model.pkl
   - models/scaler.pkl
   - api/main.py
   - api/Dockerfile
   - api/client_example.py

🚀 To run the API:
   source .venv/bin/activate
   pip install fastapi uvicorn
   uvicorn api.main:app --reload --port 8000

📖 API Documentation:
   http://localhost:8000/docs
""")

print("\n✅ Phase 3 Deployment Complete!")
